In [ ]:
import torch.nn as nn
import torch


class Code(nn.Identity): ...

In [ ]:
# Каркас AE: заполните отмеченные места.


class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 16) -> None:
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: два свёрточных блока уменьшают разрешение 28 -> 14 -> 7, каналы увеличиваются 1 -> 32 -> 64. После них идёт nn.Flatten и линейный слой.
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(1),
            nn.Linear(64 * 7 * 7, latent_dim),
        )

        # Используем вспомогательный модуль для корректного отображения с помощью visualtorch.
        self.code = Code()

        # Decoder: разворачивает вектор в карту признаков. В обратном порядке применяет nn.ConvTranspose2d: разрешение увеличивается 7 -> 14 -> 28, каналы уменьшаются 64 -> 32 -> 1.
        self.decoder_conv = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.Unflatten(1, unflattened_size=(64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder_conv(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder_conv(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        code = self.code(self.encode(x))
        return self.decode(code)